In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install --no-deps unsloth

    

In [3]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)


==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.51.1.
   \\   /|    NVIDIA A10G. Num GPUs = 1. Max memory: 22.191 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.6. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "embed_tokens", "lm_head"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
    use_rslora = True,
)


Unsloth: Offloading input_embeddings to disk to save VRAM
Unsloth: Offloading output_embeddings to disk to save VRAM


Unsloth 2025.3.19 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM


In [5]:
from datasets import load_dataset

dataset = load_dataset("open-web-math/open-web-math", split="train[:2000]")  # start small
EOS_TOKEN = tokenizer.eos_token

def format_example(examples):
    return {"text": [ex + EOS_TOKEN for ex in examples["text"]]}

dataset = dataset.map(format_example, batched=True)


Resolving data files:   0%|          | 0/114 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/6315233 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=2048)

tokenized_dataset = dataset.map(tokenize, batched=True)


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [12]:
from unsloth import UnslothTrainer

class CustomTrainer(UnslothTrainer):  # subclass UnslothTrainer, not Trainer
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        logic_words = ["because", "therefore", "hence", "implies", "at least one"]
        token_ids = set()
        for word in logic_words:
            tokens = tokenizer.tokenize(word)
            ids = tokenizer.convert_tokens_to_ids(tokens)
            token_ids.update(ids)

        logic_mask = torch.zeros_like(labels, dtype=torch.float)
        for tid in token_ids:
            logic_mask += (labels == tid).float()

        α = 5.0
        soft_weights = 1.0 + torch.sigmoid(α * (logic_mask - 0.5)).to(labels.device)

        ce_loss = nn.CrossEntropyLoss(reduction="none")
        loss = ce_loss(logits.view(-1, logits.size(-1)), labels.view(-1))
        loss = loss * soft_weights.view(-1)
        loss = loss.mean()

        return (loss, outputs) if return_outputs else loss


In [13]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_steps=100,
    save_total_limit=1,
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


In [15]:
import os
os.environ['UNSLOTH_RETURN_LOGITS'] = '1'

trainer.train()


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 570,425,344/1,000,000,000 (57.04% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,53.077600
10,43.874500
15,25.239600
20,11.705900
25,3.522100
30,0.444100
35,0.100100
40,0.054700
45,0.045200
50,0.033800


TrainOutput(global_step=125, training_loss=5.527145215447992, metrics={'train_runtime': 846.8027, 'train_samples_per_second': 2.362, 'train_steps_per_second': 0.148, 'total_flos': 3.7934812102656e+16, 'train_loss': 5.527145215447992})

In [16]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)
prompt = "If x is greater than 5 and y is less than x, then it implies"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

streamer = TextStreamer(tokenizer)
model.generate(**inputs, streamer=streamer, max_new_tokens=100)


<|begin_of_text|>If x is greater than 5 and y is less than x, then it implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies implies


tensor([[128000,   2746,    865,    374,   7191,   1109,    220,     20,    323,
            379,    374,   2753,   1109,    865,     11,   1243,    433,  24897,
          24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,
          24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,
          24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,
          24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,
          24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,
          24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,
          24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,
          24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,
          24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,
          24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,  24897,
          24897,  24897,  24

In [20]:

model.save_pretrained("saved_lora_model")
tokenizer.save_pretrained("saved_lora_model")



('saved_lora_model/tokenizer_config.json',
 'saved_lora_model/special_tokens_map.json',
 'saved_lora_model/tokenizer.json')